In [1]:
import pyarrow.parquet as pq
import pandas as pd
import os

In [4]:
base_path = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\filtered"
output_dir = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\merged"
filtered_measurements_file  = os.path.join(base_path, "filtered_measurements_encoded.parquet")
filtered_bookings_file = os.path.join(base_path, "filtered_bookings.parquet")

## Load the filtered Parquet files

In [3]:
bookings_df = pq.read_table(filtered_bookings_file).to_pandas()

In [5]:
measurements_df = pq.read_table(filtered_measurements_file).to_pandas()

## Merge with respect to the creation time

In [6]:
# Convert to datetime
measurements_df['created_at'] = pd.to_datetime(measurements_df['created_at'], errors='coerce')
bookings_df['created_at'] = pd.to_datetime(bookings_df['created_at'], errors='coerce')

In [7]:
# Determine overlapping time window
start_time = max(measurements_df['created_at'].min(), bookings_df['created_at'].min())
end_time = min(measurements_df['created_at'].max(), bookings_df['created_at'].max())

In [8]:
# Filter for overlapping window
filtered_measurements = measurements_df[
    (measurements_df['created_at'] >= start_time) & (measurements_df['created_at'] <= end_time)
]
filtered_bookings = bookings_df[
    (bookings_df['created_at'] >= start_time) & (bookings_df['created_at'] <= end_time)
]

In [9]:
# Merge on booking_id
merged_df = pd.merge(filtered_measurements, filtered_bookings, on="booking_id", suffixes=('_meas', '_book'))

In [10]:
# Drop booking_id column after merge
merged_df = merged_df.drop(columns=['booking_id'])

In [11]:
merged_df.head()

,measure_step_number,measure_value,created_at_meas,book_state_meas,part_number,serial_number_id_meas,station_id_meas,lower_limit,upper_limit,measurement_name_encoded,measurement_unit_encoded,is_within_limits,book_state_book,workstep_number_mes,created_at_book,book_stamp,serial_number_id_book,station_id_book,part_group,line_id
0,273,4.11456,2025-03-28 10:43:05.672000+00:00,1,a13143e7,32e85784,464416bb,0.0,10.0,41832,2581331,1,1,2,2025-03-28 10:43:04.158000+00:00,2025-03-28 10:43:05.759000+00:00,32e85784,464416bb,83e223f1,ad63c958
1,863,828.67300,2025-03-11 23:55:49.788000+00:00,1,5a6867de,c89a5b10,464416bb,490.0,910.0,41833,13027095,1,1,2,2025-03-11 23:55:49.569000+00:00,2025-03-11 23:55:49.840000+00:00,c89a5b10,464416bb,8db45195,ad63c958
2,863,830.37200,2025-03-17 21:03:22.172000+00:00,1,5a6867de,4fdb5292,464416bb,490.0,910.0,41833,13027095,1,1,2,2025-03-17 21:03:21.912000+00:00,2025-03-17 21:03:22.216000+00:00,4fdb5292,464416bb,8db45195,ad63c958
3,863,830.42200,2025-05-06 22:10:04.760000+00:00,1,5a6867de,aa69f270,464416bb,490.0,910.0,41833,13027095,1,1,2,2025-05-06 22:10:04.377000+00:00,2025-05-06 22:10:04.832000+00:00,aa69f270,464416bb,8db45195,ad63c958
4,863,829.61300,2025-04-22 08:12:20.181000+00:00,1,5a6867de,1d99775f,464416bb,490.0,910.0,41833,13027095,1,1,2,2025-04-22 08:12:19.926000+00:00,2025-04-22 08:12:20.228000+00:00,1d99775f,464416bb,8db45195,ad63c958


In [12]:
# Save merged result
merged_output_path = os.path.join(output_dir, "merged_bookings_measurements.parquet")
merged_df.to_parquet(merged_output_path, index=False)

print("Merged file saved to:", merged_output_path)
print("Merged shape:", merged_df.shape)
print("Time window:", start_time, "to", end_time)

Merged file saved to: M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\merged\merged_bookings_measurements.parquet
Merged shape: (43934134, 20)
Time window: 2025-03-01 01:21:20.773000+00:00 to 2025-05-14 01:02:06.180000+00:00
